In [18]:
from pathlib import Path
import math
import re
import torch
from src.tokenization.bpe import bpe_file_sha256

ckpts = sorted(Path("checkpoints/lab4_gqa").glob("final*.ckpt"))
print("Found checkpoints:")
for ckpt in ckpts:
    print(ckpt)
assert ckpts, "No final checkpoint found. Run training first."

def checkpoint_perplexity(path: Path) -> float:
    match = re.search(r"val_perplexity=([0-9]+(?:\.[0-9]+)?)\.ckpt$", path.name)
    return float(match.group(1)) if match else math.inf

BEST_CKPT = str(min(ckpts, key=checkpoint_perplexity))
checkpoint_payload = torch.load(BEST_CKPT, map_location="cpu", weights_only=False)
checkpoint_paths = checkpoint_payload.get("hyper_parameters", {}).get("paths", {})
EXPECTED_TOKENIZER_SHA256 = checkpoint_paths.get("tokenizer_sha256")

print("BEST_CKPT=", BEST_CKPT)
print("BEST_VAL_PERPLEXITY=", checkpoint_perplexity(Path(BEST_CKPT)))
print("EXPECTED_TOKENIZER_SHA256=", EXPECTED_TOKENIZER_SHA256)


Found checkpoints:
checkpoints/lab4_gqa/final-epoch=02-val_perplexity=3.85.ckpt
BEST_CKPT= checkpoints/lab4_gqa/final-epoch=02-val_perplexity=3.85.ckpt
BEST_VAL_PERPLEXITY= 3.85
EXPECTED_TOKENIZER_SHA256= fb8c215480ab23f492e5869d807b57337352fa8f4abde8086a5ed8a36ee33d2b


In [19]:
from pathlib import Path
from src.tokenization.bpe import bpe_file_sha256

candidate_tokenizers = [
    Path("data/processed/common_crawl_bpe.json"),
    Path("common_crawl_bpe.json"),
    Path("checkpoints/lab4_gqa/common_crawl_bpe.json"),
    Path("artifacts/common_crawl_bpe.json"),
]

TOKENIZER_PATH = None
print("Tokenizer candidates:")
for candidate in candidate_tokenizers:
    if not candidate.exists():
        print("missing", candidate)
        continue
    digest = bpe_file_sha256(candidate)
    marker = "MATCH" if digest == EXPECTED_TOKENIZER_SHA256 else "mismatch"
    print(marker, candidate, digest)
    if digest == EXPECTED_TOKENIZER_SHA256:
        TOKENIZER_PATH = str(candidate)

if TOKENIZER_PATH is None:
    raise FileNotFoundError(
        "No tokenizer matching this checkpoint was found. "
        "Copy the common_crawl_bpe.json saved together with the checkpoint into "
        "data/processed/common_crawl_bpe.json or checkpoints/lab4_gqa/common_crawl_bpe.json. "
        f"Expected sha256: {EXPECTED_TOKENIZER_SHA256}"
    )

print("TOKENIZER_PATH=", TOKENIZER_PATH)


Tokenizer candidates:
mismatch data/processed/common_crawl_bpe.json bb2a79c378411bfd4a64b28f1fd12a96ecef8ea50c0c341351b4f48dbf7dfe5b
missing common_crawl_bpe.json
MATCH checkpoints/lab4_gqa/common_crawl_bpe.json fb8c215480ab23f492e5869d807b57337352fa8f4abde8086a5ed8a36ee33d2b
missing artifacts/common_crawl_bpe.json
TOKENIZER_PATH= checkpoints/lab4_gqa/common_crawl_bpe.json


In [20]:
prompt = "The history of artificial intelligence"

!python -m cli.lab4 generate --config configs/lab4_gqa.yaml --checkpoint "$BEST_CKPT" --tokenizer "$TOKENIZER_PATH" --prompt "$prompt" --max-new-tokens 80 --temperature 0.6 --top-k 10


/Users/bogdanroshchupkin/Downloads/modern-ai-architecture/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.6.5, which is newer than your current Lightning version: v2.6.1
The history of artificial intelligence is a state of the battle in the contract of the subject , and that the family co


In [21]:
prompt = "The history of artificial intelligence"

!python -m cli.lab4 generate --config configs/lab4_gqa.yaml --checkpoint "$BEST_CKPT" --tokenizer "$TOKENIZER_PATH" --prompt "$prompt" --max-new-tokens 80 --temperature 0.6 --top-k 10 --no-kv-cache


/Users/bogdanroshchupkin/Downloads/modern-ai-architecture/.venv/lib/python3.11/site-packages/lightning/pytorch/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.6.5, which is newer than your current Lightning version: v2.6.1
The history of artificial intelligence between 2009 and 1988 , at the ship of which had been intended as " the ship " s


In [22]:
import torch
from cli.lab2 import load_config
from src.training.lightning_module import GPTLightningModule
from src.tokenization.bpe import BpeTokenizer

config = load_config("configs/lab4_gqa.yaml")
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = BpeTokenizer.load(TOKENIZER_PATH)
model = GPTLightningModule.load_from_checkpoint(BEST_CKPT, config=config).to(device)
model.eval()

prompt = "The history of artificial intelligence"
input_ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
segment_ids = torch.ones_like(input_ids)
with torch.no_grad():
    logits = model(input_ids, segment_ids)[:, -1, :]
    probs = torch.softmax(logits, dim=-1)
    values, indices = torch.topk(probs, k=20, dim=-1)

inverse_vocab = {idx: token for token, idx in tokenizer.stoi.items()}
print("Top next-token probabilities:")
for value, index in zip(values[0].tolist(), indices[0].tolist()):
    print(f"{index:4d} {inverse_vocab.get(index, '<unk>')!r:18s} {value:.4f}")


Top next-token probabilities:
  81 'o'                0.1626
  75 'i'                0.1350
  89 'w'                0.1180
  67 'a'                0.1008
  13 ','                0.0912
  86 't'                0.0641
  72 'f'                0.0431
  69 'c'                0.0379
  68 'b'                0.0335
  85 's'                0.0259
  84 'r'                0.0182
  70 'd'                0.0174
  82 'p'                0.0172
  74 'h'                0.0156
  79 'm'                0.0134
  15 '.'                0.0131
   9 '('                0.0127
  71 'e'                0.0105
  73 'g'                0.0079
  78 'l'                0.0075
